# Kaggle S6E9 

This notebook tests the current Kaggle discussion idea of **replication-aware split selection** without replacing the established ensemble. Candidate splits are scored using ordinary Newton gain plus stability across random replication subgroups.

The implementation is deliberately bounded: by default it runs a **single fixed outer validation fold** on a capped screening sample, then reports whether the new booster is worth a full 3-fold confirmation. It does not perform an unbounded hyperparameter or blend search.

Frozen benchmark: **0.945454 OOF rank-ensemble AUC**. The new model only needs to provide complementary ranking or a meaningful single-fold gain to justify a confirmation run.

In [1]:
# Configuration
import os, warnings, random, math, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb

SEED = 42
N_OUTER_FOLDS = 3
N_INNER_FOLDS = 4
SMOOTHING_VALUES = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]

# Replication-aware booster controls. These are intentionally modest so the
# screening run remains bounded.
REPLICATION_WEIGHT = 0.50
N_REPLICATIONS = 8
MAX_DEPTH = 3
N_ESTIMATORS = 80
LEARNING_RATE = 0.05
L2_REG = 5.0
MIN_NODE_SAMPLES = 200
N_THRESHOLDS = 16
SCREEN_ROWS = 60000
FEATURE_LIMIT = 18
MIN_GAIN = 1e-10
# A reproducible hard cap for work at every node.
NODE_SAMPLE_CAP = 45000

TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e9/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e9/test.csv"
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = "train.csv"
    TEST_PATH = "test.csv"

TARGET = "Will_Buy_EV"
ID_COL = "id"
CURRENT_CHAMPION = 0.945454

np.random.seed(SEED)
random.seed(SEED)
print("Train path:", TRAIN_PATH)
print("Current ensemble benchmark:", CURRENT_CHAMPION)
print("Replication weight:", REPLICATION_WEIGHT)
print("Replications:", N_REPLICATIONS)


Train path: /kaggle/input/competitions/playground-series-s6e9/train.csv
Current ensemble benchmark: 0.945454
Replication weight: 0.5
Replications: 8


In [2]:
# Load data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train[TARGET].astype(str) == "Yes").astype(np.int8).values
print("train:", train.shape)
print("test :", test.shape)
print("target mean:", y.mean())


train: (668665, 15)
test : (286571, 14)
target mean: 0.17464500160768098


## 2. Reuse the proven adaptive exact-value TE feature recipe

The replication-aware booster is tested on the same feature representation that produced the current strong models: raw numeric features, conservative engineered features, frequency features, digit features, and true OOF exact-value target encoding with per-column smoothing selected only inside the outer training fold.

In [3]:
CAT_COLS = [
    "Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
    "Subsidy_Available", "Range_Anxiety_Level"
]
TE_COLS = [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    "Environmental_Concern_Level", "Number_of_Cars_Owned"
]
NUMERIC_VALUE_COLS = [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Number_of_Cars_Owned", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work",
    "Environmental_Concern_Level"
]

def base_features(df):
    out = pd.DataFrame(index=df.index)
    for c in NUMERIC_VALUE_COLS:
        out[c] = pd.to_numeric(df[c], errors="coerce")
    out["income_100k"] = out["Annual_Income_USD"] / 100000.0
    out["concern_ord"] = out["Environmental_Concern_Level"].astype(float)
    out["anxiety_ord"] = df["Range_Anxiety_Level"].map({"Low":0,"Medium":1,"High":2}).astype(float)
    out["subsidy_bin"] = df["Subsidy_Available"].map({"No":0,"Yes":1}).astype(float)
    out["home_charge_bin"] = df["Home_Charging_Possible"].map({"No":0,"Yes":1}).astype(float)
    out["income_x_concern"] = out["income_100k"] * out["concern_ord"]
    out["income_x_subsidy"] = out["income_100k"] * out["subsidy_bin"]
    out["concern_x_subsidy"] = out["concern_ord"] * out["subsidy_bin"]
    out["total_charging"] = out["Charging_Stations_Near_Home"] + out["Charging_Stations_Near_Work"]
    out["commute_per_charge"] = out["Daily_Commute_km"] / (out["total_charging"] + 1.0)
    for c in CAT_COLS:
        out[c] = df[c].astype(str)
    return out

def fit_te_mapping(series, target, smoothing):
    tmp = pd.DataFrame({"x": series.values, "y": target})
    stats = tmp.groupby("x", dropna=False)["y"].agg(["sum", "count"])
    gm = float(tmp["y"].mean())
    mapping = (stats["sum"] + smoothing * gm) / (stats["count"] + smoothing)
    return mapping, gm

def apply_te_mapping(series, mapping, gm):
    return series.map(mapping).fillna(gm).astype(np.float32).values

def select_smoothing(train_df, target, outer_train_idx, seed):
    inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed)
    yo = target[outer_train_idx]
    best = {}
    for c in TE_COLS:
        best_auc = -np.inf
        best_s = 20.0
        for s in SMOOTHING_VALUES:
            pred = np.zeros(len(outer_train_idx), dtype=np.float32)
            for itp, ivp in inner.split(np.zeros(len(yo)), yo):
                itg = outer_train_idx[itp]
                ivg = outer_train_idx[ivp]
                mp, gm = fit_te_mapping(train_df[c].iloc[itg], target[itg], s)
                pred[ivp] = apply_te_mapping(train_df[c].iloc[ivg], mp, gm)
            auc = roc_auc_score(yo, pred)
            if auc > best_auc:
                best_auc, best_s = auc, float(s)
        best[c] = best_s
        print(f"  {c:35s} best_smoothing={best_s:6.1f} inner_AUC={best_auc:.6f}")
    return best

def make_true_oof_te_adaptive(train_df, target, outer_train_idx, outer_valid_idx, test_df, smoothing_by_col, seed):
    n_tr = len(outer_train_idx)
    te_train = pd.DataFrame(index=np.arange(n_tr), columns=[f"{c}_te" for c in TE_COLS], dtype=np.float32)
    inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=seed)
    yo = target[outer_train_idx]
    for c in TE_COLS:
        s = smoothing_by_col[c]
        for itp, ivp in inner.split(np.zeros(n_tr), yo):
            itg = outer_train_idx[itp]
            mp, gm = fit_te_mapping(train_df[c].iloc[itg], target[itg], s)
            te_train.loc[ivp, f"{c}_te"] = apply_te_mapping(train_df[c].iloc[outer_train_idx[ivp]], mp, gm)
        mp, gm = fit_te_mapping(train_df[c].iloc[outer_train_idx], target[outer_train_idx], s)
        va = apply_te_mapping(train_df[c].iloc[outer_valid_idx], mp, gm)
        te = apply_te_mapping(test_df[c], mp, gm)
        yield c, te_train[f"{c}_te"].values, va, te

def add_frequency_features(out, df, reference_df):
    out = out.copy()
    for c in TE_COLS:
        freq = reference_df[c].value_counts(dropna=False)
        out[f"{c}_freq"] = df[c].map(freq).fillna(0).astype(np.float32)
    return out

def add_digit_features(out, df):
    out = out.copy()
    for c in ["Annual_Income_USD", "Daily_Commute_km", "Age", "Charging_Stations_Near_Home", "Charging_Stations_Near_Work"]:
        x = pd.to_numeric(df[c], errors="coerce").fillna(-1).astype(np.int64)
        out[f"{c}_mod10"] = (x % 10).astype(np.float32)
        out[f"{c}_mod100"] = (x % 100).astype(np.float32)
    return out

def prepare_fold_features(tr_idx, va_idx, seed):
    tr_df = train.iloc[tr_idx].reset_index(drop=True)
    va_df = train.iloc[va_idx].reset_index(drop=True)
    smoothing = select_smoothing(train, y, tr_idx, seed)
    Xtr = base_features(tr_df)
    Xva = base_features(va_df)
    Xtr = add_frequency_features(Xtr, tr_df, tr_df)
    Xva = add_frequency_features(Xva, va_df, tr_df)
    Xtr = add_digit_features(Xtr, tr_df)
    Xva = add_digit_features(Xva, va_df)
    for c, a, b, _ in make_true_oof_te_adaptive(train, y, tr_idx, va_idx, test, smoothing, seed):
        Xtr[f"{c}_te"] = a
        Xva[f"{c}_te"] = b
    # Stable one-hot encoding for the custom numeric booster.
    combo = pd.concat([Xtr, Xva], axis=0, ignore_index=True)
    combo = pd.get_dummies(combo, columns=CAT_COLS, dtype=np.float32)
    Xtr = combo.iloc[:len(Xtr)].copy()
    Xva = combo.iloc[len(Xtr):].copy()
    for c in combo.columns:
        if c not in Xtr.columns:
            Xtr[c] = 0.0
        if c not in Xva.columns:
            Xva[c] = 0.0
    Xtr = Xtr.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)
    Xva = Xva.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)
    return Xtr.reset_index(drop=True), Xva.reset_index(drop=True), smoothing


## 3. Fast replication-aware Newton tree

The tree uses ordinary binary-logistic Newton gradients and Hessians. For each candidate split, it computes: (1) global Newton gain, and (2) gain stability across random replication subgroups. The final candidate score is a convex combination controlled by `REPLICATION_WEIGHT`.

To keep runtime bounded, split candidates are evaluated on a capped screening sample and at most 16 quantile thresholds per feature/node. Leaf values are then computed from the full outer-training data using the selected tree structure.

In [4]:
def _logistic_grad_hess(p, yv):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    g = (p - yv).astype(np.float64)
    h = (p * (1.0 - p)).astype(np.float64)
    return g, h

def _candidate_threshold_positions(x, n_thresholds):
    x = np.asarray(x, dtype=np.float64)
    finite = np.isfinite(x)
    if finite.sum() < 2:
        return []
    xf = x[finite]
    qs = np.linspace(0.02, 0.98, n_thresholds)
    vals = np.unique(np.quantile(xf, qs))
    vals = vals[np.isfinite(vals)]
    return vals.tolist()

def _evaluate_feature_replication(x, g, h, reps, n_thresholds):
    """Return best candidate for one feature on one node."""
    x = np.asarray(x, dtype=np.float64)
    g = np.asarray(g, dtype=np.float64)
    h = np.asarray(h, dtype=np.float64)
    reps = np.asarray(reps, dtype=np.int16)
    finite = np.isfinite(x)
    if finite.sum() < MIN_NODE_SAMPLES:
        return None
    order = np.argsort(x[finite], kind="mergesort")
    xs = x[finite][order]
    gs = g[finite][order]
    hs = h[finite][order]
    rs = reps[finite][order]
    thresholds = _candidate_threshold_positions(xs, n_thresholds)
    if not thresholds:
        return None

    total_g = float(gs.sum())
    total_h = float(hs.sum())
    if total_h <= 0:
        return None

    # Cumulative global sums.
    cg = np.cumsum(gs)
    ch = np.cumsum(hs)

    # Per-replication cumulative sums; n is capped, so this is small.
    rep_g = np.zeros((N_REPLICATIONS, len(xs)), dtype=np.float64)
    rep_h = np.zeros((N_REPLICATIONS, len(xs)), dtype=np.float64)
    for r in range(N_REPLICATIONS):
        mask = (rs == r)
        rep_g[r] = np.cumsum(np.where(mask, gs, 0.0))
        rep_h[r] = np.cumsum(np.where(mask, hs, 0.0))

    # Only use thresholds that actually separate observations.
    positions = np.searchsorted(xs, np.asarray(thresholds), side="right") - 1
    positions = np.unique(positions[(positions >= 0) & (positions < len(xs) - 1)])
    if len(positions) == 0:
        return None

    gains = []
    for pos in positions:
        gl = float(cg[pos]); hl = float(ch[pos])
        gr = total_g - gl; hr = total_h - hl
        if hl <= 1e-8 or hr <= 1e-8:
            continue
        global_gain = gl*gl/(hl + L2_REG) + gr*gr/(hr + L2_REG) - total_g*total_g/(total_h + L2_REG)

        group_gain = np.zeros(N_REPLICATIONS, dtype=np.float64)
        valid = np.zeros(N_REPLICATIONS, dtype=bool)
        for r in range(N_REPLICATIONS):
            tgl = rep_g[r, pos]; thl = rep_h[r, pos]
            tgr = rep_g[r, -1] - tgl; thr = rep_h[r, -1] - thl
            valid[r] = (thl > 1e-8) and (thr > 1e-8)
            if valid[r]:
                gt = tgl*tgl/(thl + L2_REG) + tgr*tgr/(thr + L2_REG) - (tgl+tgr)*(tgl+tgr)/(thl+thr+L2_REG)
                group_gain[r] = max(float(gt), 0.0)

        valid_fraction = float(valid.mean())
        if valid_fraction == 0:
            continue
        agreement_fraction = float((group_gain[valid] > MIN_GAIN).mean()) if valid.any() else 0.0
        gmax = float(group_gain[valid].max()) if valid.any() else 0.0
        norm_group = group_gain[valid] / (gmax + 1e-12)
        mean_rank_strength = float(norm_group.mean()) if len(norm_group) else 0.0
        rank_std_normalized = float(norm_group.std()) if len(norm_group) else 0.0
        # Match the discussion's replication-only scoring form.
        replication_score = (
            0.55 * agreement_fraction
            + 0.35 * mean_rank_strength
            + 0.10 * valid_fraction
            - 0.15 * rank_std_normalized
        )

        gains.append((
            float(global_gain), float(replication_score),
            float(thresholds[np.argmin(np.abs(np.asarray(xs[positions]) - xs[pos]))]),
            int(pos)
        ))

    if not gains:
        return None

    # Normalize global gain within this feature's candidate set.
    globs = np.asarray([z[0] for z in gains], dtype=np.float64)
    lo, hi = float(globs.min()), float(globs.max())
    best_local = None
    for gg, rscr, _, pos in gains:
        gnorm = (gg - lo) / (hi - lo + 1e-12)
        final_score = (1.0 - REPLICATION_WEIGHT) * gnorm + REPLICATION_WEIGHT * rscr
        if best_local is None or final_score > best_local[0]:
            thr = float(xs[pos])
            best_local = (final_score, gg, rscr, thr)
    return best_local

class ReplicationAwareNewtonBoosting:
    def __init__(self, n_estimators=80, learning_rate=0.05, max_depth=3,
                 n_replications=8, n_thresholds=16, random_state=42):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.n_replications = n_replications
        self.n_thresholds = n_thresholds
        self.random_state = random_state
        self.trees_ = []
        self.base_score_ = None
        self.feature_names_ = None

    def _build_tree(self, X_screen, g_screen, h_screen, rep_labels):
        n = len(X_screen)
        all_idx = np.arange(n)
        tree = []

        def recurse(indices, depth):
            node_id = len(tree)
            tree.append({"leaf": True, "weight": 0.0})
            if depth >= self.max_depth or len(indices) < MIN_NODE_SAMPLES:
                G = float(g_screen[indices].sum()); H = float(h_screen[indices].sum())
                tree[node_id] = {"leaf": True, "weight": -G/(H + L2_REG)}
                return node_id

            if len(indices) > NODE_SAMPLE_CAP:
                rng = np.random.default_rng(self.random_state + depth + node_id)
                indices_eval = rng.choice(indices, NODE_SAMPLE_CAP, replace=False)
            else:
                indices_eval = indices

            best = None
            for j in range(X_screen.shape[1]):
                res = _evaluate_feature_replication(
                    X_screen.iloc[indices_eval, j].values,
                    g_screen[indices_eval],
                    h_screen[indices_eval],
                    rep_labels[indices_eval],
                    self.n_thresholds
                )
                if res is None:
                    continue
                score, gg, rscr, thr = res
                if best is None or score > best[0]:
                    best = (score, gg, rscr, j, thr)

            if best is None or best[1] <= MIN_GAIN:
                G = float(g_screen[indices].sum()); H = float(h_screen[indices].sum())
                tree[node_id] = {"leaf": True, "weight": -G/(H + L2_REG)}
                return node_id

            _, gg, rscr, j, thr = best
            left_mask = X_screen.iloc[indices, j].values <= thr
            left = indices[left_mask]
            right = indices[~left_mask]
            if len(left) < MIN_NODE_SAMPLES or len(right) < MIN_NODE_SAMPLES:
                G = float(g_screen[indices].sum()); H = float(h_screen[indices].sum())
                tree[node_id] = {"leaf": True, "weight": -G/(H + L2_REG)}
                return node_id

            tree[node_id] = {"leaf": False, "feature": int(j), "threshold": float(thr),
                             "left": None, "right": None, "global_gain": float(gg),
                             "replication_score": float(rscr)}
            left_id = recurse(left, depth + 1)
            right_id = recurse(right, depth + 1)
            tree[node_id]["left"] = left_id
            tree[node_id]["right"] = right_id
            return node_id

        root = recurse(all_idx, 0)
        return tree, root

    def _predict_tree(self, X, tree, root):
        pred = np.zeros(len(X), dtype=np.float64)
        # Iterative traversal by node masks.
        def visit(node_id, idx):
            node = tree[node_id]
            if len(idx) == 0:
                return
            if node["leaf"]:
                pred[idx] = node["weight"]
                return
            vals = X.iloc[idx, node["feature"]].values
            lm = vals <= node["threshold"]
            visit(node["left"], idx[lm])
            visit(node["right"], idx[~lm])
        visit(root, np.arange(len(X)))
        return pred

    def fit(self, X, yv, screen_rows=None, verbose=True):
        self.feature_names_ = list(X.columns)
        X = X.reset_index(drop=True).astype(np.float32)
        yv = np.asarray(yv, dtype=np.float64)
        n = len(X)
        rng = np.random.default_rng(self.random_state)
        if screen_rows is None or screen_rows >= n:
            screen_idx = np.arange(n)
        else:
            # Stratified-ish sample through random order followed by class balancing.
            pos = np.flatnonzero(yv == 1)
            neg = np.flatnonzero(yv == 0)
            n_pos = min(len(pos), int(screen_rows * yv.mean()))
            n_neg = screen_rows - n_pos
            screen_idx = np.concatenate([
                rng.choice(pos, n_pos, replace=False),
                rng.choice(neg, n_neg, replace=False)
            ])
            rng.shuffle(screen_idx)

        Xs = X.iloc[screen_idx].reset_index(drop=True)
        ys = yv[screen_idx]
        rep_labels = rng.integers(0, self.n_replications, size=len(Xs), dtype=np.int16)

        prior = float(np.clip(ys.mean(), 1e-5, 1-1e-5))
        self.base_score_ = math.log(prior / (1-prior))
        full_margin = np.full(n, self.base_score_, dtype=np.float64)
        screen_margin = np.full(len(Xs), self.base_score_, dtype=np.float64)
        self.trees_ = []

        for it in range(self.n_estimators):
            ps = 1.0 / (1.0 + np.exp(-np.clip(screen_margin, -20, 20)))
            gs, hs = _logistic_grad_hess(ps, ys)
            tree, root = self._build_tree(Xs, gs, hs, rep_labels)
            self.trees_.append((tree, root))

            step_s = self._predict_tree(Xs, tree, root) * self.learning_rate
            screen_margin += step_s

            # Update the full training margin using the learned structure.
            step_full = self._predict_tree(X, tree, root) * self.learning_rate
            full_margin += step_full

            if verbose and ((it + 1) % 10 == 0 or it == 0):
                pfull = 1.0 / (1.0 + np.exp(-np.clip(full_margin, -20, 20)))
                # This is training AUC only for progress monitoring.
                print(f"  tree {it+1:3d}/{self.n_estimators} | train AUC={roc_auc_score(yv, pfull):.6f}")
        return self

    def predict_proba(self, X):
        X = X[self.feature_names_].reset_index(drop=True).astype(np.float32)
        margin = np.full(len(X), self.base_score_, dtype=np.float64)
        for tree, root in self.trees_:
            margin += self.learning_rate * self._predict_tree(X, tree, root)
        p = 1.0 / (1.0 + np.exp(-np.clip(margin, -20, 20)))
        return np.column_stack([1-p, p])


## 4. One-fold screen

This cell is the main test. It uses the first fixed stratified outer fold, fits the replication-aware booster, and compares it with the existing LightGBM recipe on exactly the same validation rows. By default this is the only expensive custom-booster run.

In [5]:
outer = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=SEED)
tr_idx, va_idx = next(outer.split(train, y))

print("=" * 72)
print("REPLICATION-AWARE SCREEN")
print("=" * 72)

t0 = time.time()
Xtr, Xva, smoothing = prepare_fold_features(tr_idx, va_idx, SEED + 1)
print("Feature shapes:", Xtr.shape, Xva.shape)
print("Feature count used by booster:", Xtr.shape[1])

# Keep only numeric columns and cap to the strongest known features by univariate
# AUC on the outer-training partition. This makes the custom booster practical.
feature_auc = {}
ytr = y[tr_idx]
for c in Xtr.columns:
    col = Xtr[c].values
    if np.nanstd(col) < 1e-12:
        continue
    try:
        a = roc_auc_score(ytr, col)
        feature_auc[c] = max(a, 1-a)
    except Exception:
        pass
selected_features = sorted(feature_auc, key=feature_auc.get, reverse=True)[:FEATURE_LIMIT]
Xtr_r = Xtr[selected_features].copy()
Xva_r = Xva[selected_features].copy()
print("Selected custom-booster features:")
for c in selected_features:
    print(f"  {c:40s} {feature_auc[c]:.6f}")

# Reference LightGBM on exactly the same selected numeric features.
lgbm = lgb.LGBMClassifier(
    objective="binary", n_estimators=1800, learning_rate=0.03, num_leaves=31,
    min_child_samples=80, subsample=0.85, colsample_bytree=0.85,
    reg_alpha=0.1, reg_lambda=1.0, random_state=SEED + 1, n_jobs=-1, verbosity=-1
)
lgbm.fit(Xtr_r, ytr, eval_set=[(Xva_r, y[va_idx])], eval_metric="auc",
          callbacks=[lgb.early_stopping(100, verbose=False)])
p_lgb = lgbm.predict_proba(Xva_r)[:, 1]
lgb_auc = roc_auc_score(y[va_idx], p_lgb)
print(f"\nReference LightGBM screen AUC: {lgb_auc:.6f}")

ra = ReplicationAwareNewtonBoosting(
    n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH, n_replications=N_REPLICATIONS,
    n_thresholds=N_THRESHOLDS, random_state=SEED + 101
)
ra.fit(Xtr_r, ytr, screen_rows=min(SCREEN_ROWS, len(Xtr_r)), verbose=True)
p_ra = ra.predict_proba(Xva_r)[:, 1]
ra_auc = roc_auc_score(y[va_idx], p_ra)
print(f"\nReplication-aware validation AUC: {ra_auc:.6f}")
print(f"LightGBM reference AUC:           {lgb_auc:.6f}")
print(f"Delta (RA - LGBM):                 {ra_auc - lgb_auc:+.6f}")
print(f"Elapsed screen time:               {(time.time()-t0)/60:.2f} min")


REPLICATION-AWARE SCREEN
  Age                                 best_smoothing= 100.0 inner_AUC=0.546748
  Annual_Income_USD                   best_smoothing=   5.0 inner_AUC=0.704804
  Daily_Commute_km                    best_smoothing=   2.0 inner_AUC=0.548219
  Charging_Stations_Near_Home         best_smoothing= 100.0 inner_AUC=0.522574
  Charging_Stations_Near_Work         best_smoothing=   0.5 inner_AUC=0.510721
  Environmental_Concern_Level         best_smoothing=   0.5 inner_AUC=0.843574
  Number_of_Cars_Owned                best_smoothing=   0.5 inner_AUC=0.506318
Feature shapes: (445776, 58) (222889, 58)
Feature count used by booster: 58
Selected custom-booster features:
  concern_x_subsidy                        0.915205
  income_x_concern                         0.864305
  Environmental_Concern_Level              0.844105
  concern_ord                              0.844105
  Environmental_Concern_Level_te           0.843574
  income_x_subsidy                         0.814282


## 5. OOF diversity check

If the replication-aware booster is not individually stronger, its value can still come from a different ranking. This cell reports rank correlation against the reference LightGBM and tests a **fixed 50/50 rank blend**; no large optimizer is used.

In [6]:
r_lgb = rankdata(p_lgb).astype(np.float32) / len(p_lgb)
r_ra = rankdata(p_ra).astype(np.float32) / len(p_ra)
print("Spearman rank correlation:", np.corrcoef(r_lgb, r_ra)[0,1])
blend_50 = 0.5 * r_lgb + 0.5 * r_ra
print("50/50 rank blend AUC:", roc_auc_score(y[va_idx], blend_50))
print("Reference LGBM AUC:   ", roc_auc_score(y[va_idx], r_lgb))


Spearman rank correlation: 0.9827667657316665
50/50 rank blend AUC: 0.9409534063402186
Reference LGBM AUC:    0.9418314741450515


## Decision rule

The frozen benchmark is **0.945454 OOF**. Keep the replication-aware model only if it either improves the direct validation/OOF AUC materially or adds enough ranking diversity to improve a simple rank blend. The custom booster is intentionally a bounded research test; it does not replace the established LightGBM/XGB/RealMLP ensemble unless the evidence supports doing so.

## 6. Final Kaggle Submission

The next cell trains the final model on all training data and creates `submission.csv` in the Kaggle working directory.

In [7]:
# FINAL KAGGLE SUBMISSION
# This cell trains the final LightGBM model on the full training set,
# generates predictions for the Kaggle test set, and writes submission.csv.
#
# The output follows sample_submission.csv exactly: [id, Will_Buy_EV].

print("=" * 72)
print("FINAL KAGGLE SUBMISSION")
print("=" * 72)

# Build train/test features using the same feature engineering used above.
full_train_base = base_features(train)
full_test_base = base_features(test)

# Frequency features: frequencies are learned from the full training data.
full_train_X = add_frequency_features(full_train_base, train, train)
full_test_X = add_frequency_features(full_test_base, test, train)

# Digit features.
full_train_X = add_digit_features(full_train_X, train)
full_test_X = add_digit_features(full_test_X, test)

# Target encoding: smoothing is selected with inner CV over the full training
# set; training rows get leakage-safe OOF encodings and test rows use mappings
# fitted on all training rows. This reuses the same helpers as the screen, with
# the "outer training" set being every row and an empty validation set.
all_idx = np.arange(len(train))
full_smoothing = select_smoothing(train, y, all_idx, SEED)
for c, tr_te, _, te_te in make_true_oof_te_adaptive(
        train, y, all_idx, np.array([], dtype=int), test, full_smoothing, SEED):
    full_train_X[f"{c}_te"] = tr_te
    full_test_X[f"{c}_te"] = te_te

# One-hot encode categoricals using the combined train/test columns.
combined = pd.concat([full_train_X, full_test_X], axis=0, ignore_index=True)
combined = pd.get_dummies(combined, columns=CAT_COLS, dtype=np.float32)

full_train_X = combined.iloc[:len(train)].copy()
full_test_X = combined.iloc[len(train):].copy()

full_train_X = (
    full_train_X.replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .astype(np.float32)
)
full_test_X = (
    full_test_X.replace([np.inf, -np.inf], np.nan)
    .fillna(0.0)
    .astype(np.float32)
)

# Select the same number of strongest features used by the screen.
full_feature_auc = {}
for c in full_train_X.columns:
    col = full_train_X[c].values
    if np.nanstd(col) < 1e-12:
        continue
    try:
        a = roc_auc_score(y, col)
        full_feature_auc[c] = max(a, 1.0 - a)
    except Exception:
        pass

final_features = sorted(
    full_feature_auc, key=full_feature_auc.get, reverse=True
)[:FEATURE_LIMIT]

X_final = full_train_X[final_features]
X_test_final = full_test_X[final_features]

print(f"Final feature count: {len(final_features)}")
print("Top features:")
for c in final_features:
    print(f"  {c:40s} {full_feature_auc[c]:.6f}")

# Train the final LightGBM model on ALL available training rows.
# 1800 is the same maximum used in the validation screen.
final_lgbm = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=1800,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=80,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

final_lgbm.fit(X_final, y)

test_predictions = final_lgbm.predict_proba(X_test_final)[:, 1]

# Match the supplied sample submission exactly.
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: test_predictions
})

# Sanity checks before writing.
assert len(submission) == len(test), "Submission row count does not match test.csv."
assert submission[ID_COL].equals(test[ID_COL]), "Submission IDs do not match test.csv."
assert submission[TARGET].between(0, 1).all(), "Predictions must be probabilities in [0, 1]."

submission.to_csv("submission.csv", index=False)

print("\nSubmission created successfully:")
print("  File: submission.csv")
print("  Shape:", submission.shape)
print("  Columns:", submission.columns.tolist())
print("\nFirst 5 rows:")
print(submission.head())


FINAL KAGGLE SUBMISSION
  Age                                 smoothing= 100.0 inner_OOF_AUC=0.546578
  Annual_Income_USD                   smoothing=   5.0 inner_OOF_AUC=0.711725
  Daily_Commute_km                    smoothing=   1.0 inner_OOF_AUC=0.550144
  Charging_Stations_Near_Home         smoothing= 100.0 inner_OOF_AUC=0.523176
  Charging_Stations_Near_Work         smoothing= 100.0 inner_OOF_AUC=0.511464
  Environmental_Concern_Level         smoothing=   0.5 inner_OOF_AUC=0.843018
  Number_of_Cars_Owned                smoothing=   0.5 inner_OOF_AUC=0.505602
Final feature count: 18
Top features:
  concern_x_subsidy                        0.914688
  income_x_concern                         0.863784
  Environmental_Concern_Level              0.843521
  concern_ord                              0.843521
  Environmental_Concern_Level_te           0.843018
  income_x_subsidy                         0.814065
  subsidy_bin                              0.717940
  Subsidy_Available_No      